<a href="https://colab.research.google.com/github/Rafak22/python/blob/main/VLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install docling Pillow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 58.1 MB/s eta 0:00:00
   ━━

In [2]:
# If using OpenRouter for the VLM instead of local GPU
!pip install openai -q

In [7]:
import base64
from pathlib import Path
from openai import OpenAI
from pydantic import BaseModel
from typing import List, Optional

# Define API_Key with your actual API key. For security, consider using Colab's Secrets feature.
# Example: API_Key = "YOUR_OPENROUTER_API_KEY_HERE"
from google.colab import userdata
API_Key = userdata.get('API_Key')

client = OpenAI(
    api_key=API_Key,
    base_url="https://openrouter.ai/api/v1",
)

VLM_MODEL = "google/gemini-2.0-flash-001"  # free vision model on OpenRouter

In [8]:
def image_to_base64(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def extract_from_image(image_path: str, prompt: str) -> str:
    ext = Path(image_path).suffix.lower()
    mime = "image/png" if ext == ".png" else "image/jpeg"
    b64 = image_to_base64(image_path)

    response = client.chat.completions.create(
        model=VLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
                    {"type": "text", "text": prompt},
                ],
            }
        ],
    )
    return response.choices[0].message.content

In [10]:
# --- Receipt ---
class Receipt(BaseModel):
    store_name: Optional[str]
    total_amount: Optional[str]
    date: Optional[str]
    items: List[str]

prompt_receipt = """
Extract the following from this receipt and return as JSON:
- store_name
- total_amount
- date
- items (list of purchased items)
Return only valid JSON, no explanation.
"""

from google.colab import files
print("Upload receipt.png")
uploaded = files.upload()
receipt_path = list(uploaded.keys())[0]

raw = extract_from_image(receipt_path, prompt_receipt)
print(raw)

Upload receipt.png


Saving فاتورة.jpg to فاتورة.jpg
```json
{
  "store_name": "TAZA COMPANY LTD",
  "total_amount": "65.00",
  "date": "21/03/2018",
  "items": [
    {
      "name": "طازة بروست",
      "quantity": 2
    },
    {
      "name": "برجر دجاج",
      "quantity": 3
    },
    {
      "name": "ساندويتش مسحب دجاج",
      "quantity": 2
    }
  ]
}
```


In [11]:
# --- Resume (PDF → image first) ---
!pip install pdf2image -q
!apt-get install -y poppler-utils -q

from pdf2image import convert_from_path
import tempfile, os

class Resume(BaseModel):
    candidate_name: Optional[str]
    email: Optional[str]
    skills: List[str]
    experience: List[str]
    education: Optional[str]

prompt_resume = """
Extract the following from this resume and return as JSON:
- candidate_name
- email
- skills (list)
- experience (list)
- education
Return only valid JSON, no explanation.
"""

print("Upload resume.pdf")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

images = convert_from_path(pdf_path)
with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
    images[0].save(tmp.name)
    raw = extract_from_image(tmp.name, prompt_resume)
    os.unlink(tmp.name)

print(raw)

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (283 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
Upload resume.pdf


Saving CV_Rafa Alshareef.pdf to CV_Rafa Alshareef.pdf
```json
{
  "candidate_name": "Rafa Alshareef",
  "email": "rafa.alshareef.dev@gmail.com",
  "skills": [
    "Programming: Python, Java, C++",
    "AI & ML: Machine Learning, NLP,",
    "Computer Vision, Agentic Al Systems",
    "Frameworks: FastAPI, LangChain",
    "Automation: n8n, Make",
    "Databases: SQL & Database Management",
    "Tools: Git & Linux",
    "Soft Skills: Communication, Adaptability,",
    "Problem-Solving, Teamwork, Time",
    "Management"
  ],
  "experience": [
    {
      "title": "Applied Al Bootcamp | SDAIA",
      "date": "March 2026 - Expected May 2026"
    },
    {
      "title": "Al Specialist | NEXTA",
      "date": "Sep 2025- Jan 2026",
      "description": "Designing and building Al demos and proof-of-concept solutions to\nshowcase Al capabilities for business use cases.\nBuilding and improving Al agents, including onboarding flows,\nconversational memory, and reasoning logic.\nConducting applied R&

In [2]:
#task2Task 2 — Arabic Document Processing with QARI-OCR
!pip install transformers accelerate Pillow pdf2image -q
!apt-get install -y poppler-utils -q

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (397 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

MODEL_ID = "NAMAA-Space/Qari-OCR-v0.3-VL-2B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [4]:
from PIL import Image
from pdf2image import convert_from_path
import os

def ocr_arabic_image(image: Image.Image) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "استخرج النص من هذه الصورة مع الحفاظ على التنسيق والهيكل."},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=1024)

    return processor.decode(output[0], skip_special_tokens=True)


def process_file(path: str) -> str:
    ext = Path(path).suffix.lower()

    if ext in [".jpg", ".jpeg", ".png"]:
        image = Image.open(path).convert("RGB")
        return ocr_arabic_image(image)

    elif ext == ".pdf":
        images = convert_from_path(path)
        results = []
        for i, img in enumerate(images):
            print(f"  Processing page {i+1}/{len(images)}...")
            results.append(ocr_arabic_image(img))
        return "\n\n".join(results)

    else:
        return f"Unsupported format: {ext}"

In [5]:
# --- Single file ---
from google.colab import files
from pathlib import Path

print("Upload a file (image or PDF)")
uploaded = files.upload()
path = list(uploaded.keys())[0]

print(f"\nProcessing: {path}")
result = process_file(path)
print(result)

Upload a file (image or PDF)


Saving فاتورة_غير_واضحة.jpg to فاتورة_غير_واضحة.jpg

Processing: فاتورة_غير_واضحة.jpg
system
You are a helpful assistant.
user
استخرج النص من هذه الصورة مع الحفاظ على التنسيق والهيكل.
assistant
<h3><i>حليب</i> <i>المراعي</i></h3><br><p><b>تونالايت</b> قودي</p><br><h3>GOODY TUNA LT MEAT 7.50 تونالايت قودي</h3><br><h3>GOODY TUNA LT MEAT 7.50 تونالايت قودي</h3><br><h3>GOODY TUNA LT MEAT 7.50 تونالايت قودي</h3><br><h3>GOODY TUNA LT MEAT 7.50 تونالايت قودي</h3><br><h3>GOODY TUNA LT MEAT 7.50 تونالايت قودي</h3><br><h3>GOODY TUNA LT MEAT 7.50 تونالايت قودي</h3><br><h3>GOODY TUNA LT MEAT 7.50 تونالايت قودي</h3><br><h3>GOODY TUNA LT MEAT 7.50 تونالايت قودي</h3><br><h3>GOODY TUNA LT MEAT 64.48 BAL</h3><br><h3>64.48</h3>
